In [ ]:
import json
import spacy
import csv
from typing import List, Dict

# Load a spaCy model for tokenization
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading spaCy model 'en_core_web_sm'. Please wait...")
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

def spacy_to_csv(spacy_data: List, id_map: Dict, output_file: str):
    """
    Converts a list of SpaCy-style annotations to a CSV file.
    
    Args:
        spacy_data: A list of annotations in the format [text, {"entities": [...]}]
        id_map: A dictionary mapping line indices to original message IDs.
        output_file: The path to the output .csv file.
    """
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['message_id', 'token', 'tag'])
        
        for idx, (text, annotations) in enumerate(spacy_data):
            original_id_info = id_map.get(str(idx), None)
            if original_id_info is None:
                print(f"Warning: No message ID found for index {idx}. Skipping.")
                continue
            original_message_id = original_id_info['original_message_id']
            
            doc = nlp(text)
            tags = ['O'] * len(doc)
            
            entities = annotations.get("entities", [])
            for start_char, end_char, label in entities:
                span = doc.char_span(start_char, end_char, label=label)
                if span is None:
                    continue
                
                if len(span) == 1:
                    tags[span.start] = f"S-{label}"
                else:
                    tags[span.start] = f"B-{label}"
                    for i in range(span.start + 1, span.end - 1):
                        tags[i] = f"I-{label}"
                    tags[span.end - 1] = f"E-{label}"
            
            for token, tag in zip(doc, tags):
                writer.writerow([original_message_id, token.text, tag])

# --- Usage ---
# 1. Replace with your file names.
input_json_file = 'annotations.json'
id_map_file = 'message_id_map.json'
output_csv_file = 'annotated_data.csv'

with open(input_json_file, 'r', encoding='utf-8') as f:
    spacy_annotations = json.load(f)

with open(id_map_file, 'r', encoding='utf-8') as f:
    message_id_map = json.load(f)

spacy_to_csv(spacy_annotations, message_id_map, output_csv_file)

print(f"Conversion complete. CSV data saved to {output_csv_file}")